Getting the data

In [1]:
from pathlib import Path

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mutual_info_score, accuracy_score


data_path = Path("course_lead_scoring_2026.csv")
if not data_path.exists():
    data_path = Path("03-classification/03-homework/course_lead_scoring_2026.csv")

df = pd.read_csv(data_path)
print(df.head(20))

       lead_source       industry employment_status       location  \
0   organic_search     technology          employed         europe   
1     social_media     technology          employed  south_america   
2         referral         retail          employed         europe   
3         paid_ads  manufacturing           student            NaN   
4         referral  manufacturing           student  north_america   
5         referral  manufacturing          employed         africa   
6              NaN      education          employed  north_america   
7         paid_ads     technology          employed  north_america   
8   organic_search     healthcare          employed           asia   
9         referral         retail           student           asia   
10        referral         retail               NaN         europe   
11        paid_ads     technology          employed           asia   
12    social_media            NaN          employed         africa   
13        paid_ads  

Data preparation

In [2]:
print("Data frame information:")
df.info()

print("Missing values:")
print(df.isnull().sum())

categorical = ["lead_source", "industry", "employment_status", "location"]
numerical = ["annual_income", "number_of_courses_viewed", "interaction_count", "lead_score"]

df[categorical] = df[categorical].fillna("NA")
df[numerical] = df[numerical].fillna(0.0)

print("Remaining missing values:")
print(df.isnull().sum())

Data frame information:
<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   lead_source               4852 non-null   str    
 1   industry                  4760 non-null   str    
 2   employment_status         4806 non-null   str    
 3   location                  4792 non-null   str    
 4   annual_income             4631 non-null   float64
 5   number_of_courses_viewed  5000 non-null   int64  
 6   interaction_count         5000 non-null   int64  
 7   lead_score                4965 non-null   float64
 8   converted                 5000 non-null   int64  
dtypes: float64(2), int64(3), str(4)
memory usage: 351.7 KB
Missing values:
lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interact

Q1. Mode for industry

In [3]:
print("Industry counts:")
print(df["industry"].value_counts())

print("Most frequent industry:")
print(df["industry"].mode()[0])

# Technology appears 1173 times.
# Answer: technology

Industry counts:
industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64
Most frequent industry:
technology


Q2. Biggest correlation

In [4]:
correlation = df[numerical].corr()
print("Correlation matrix:")
print(correlation)

pairs = [
    ("interaction_count", "lead_score"),
    ("number_of_courses_viewed", "lead_score"),
    ("number_of_courses_viewed", "interaction_count"),
    ("annual_income", "interaction_count"),
]

print("Correlations for the pairs in the question:")
for first, second in pairs:
    print(first, "and", second, correlation.loc[first, second])

# The biggest correlation is about 0.916.
# Answer: interaction_count and lead_score

Correlation matrix:
                          annual_income  number_of_courses_viewed  \
annual_income                  1.000000                  0.161300   
number_of_courses_viewed       0.161300                  1.000000   
interaction_count              0.122842                  0.721609   
lead_score                     0.229496                  0.757204   

                          interaction_count  lead_score  
annual_income                      0.122842    0.229496  
number_of_courses_viewed           0.721609    0.757204  
interaction_count                  1.000000    0.915746  
lead_score                         0.915746    1.000000  
Correlations for the pairs in the question:
interaction_count and lead_score 0.9157457980418697
number_of_courses_viewed and lead_score 0.7572042392104886
number_of_courses_viewed and interaction_count 0.7216090038937035
annual_income and interaction_count 0.12284235135955959


Splitting the data

In [5]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train["converted"].values
y_val = df_val["converted"].values
y_test = df_test["converted"].values

del df_train["converted"]
del df_val["converted"]
del df_test["converted"]
del df_full_train["converted"]

print("Training, validation and test sizes:")
print(len(df_train), len(df_val), len(df_test))

Training, validation and test sizes:
3000 1000 1000


Q3. Biggest mutual information score

In [6]:
# Use only the training data.
mi_scores = {}
for column in categorical:
    score = mutual_info_score(y_train, df_train[column])
    mi_scores[column] = round(score, 2)

print("Mutual information scores:")
print(pd.Series(mi_scores).sort_values(ascending=False))

# Lead source has the biggest score: 0.03.
# Answer: lead_source

Mutual information scores:
lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64


Q4. Logistic regression accuracy

In [7]:
train_dicts = df_train.to_dict(orient="records")
val_dicts = df_val.to_dict(orient="records")

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

model = LogisticRegression(solver="liblinear", C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict_proba(X_val)[:, 1]
converted_pred = y_pred >= 0.5
original_accuracy = accuracy_score(y_val, converted_pred)

print("Validation accuracy:")
print(original_accuracy)
print("Rounded accuracy:")
print(round(original_accuracy, 2))

# Accuracy is 0.645 before rounding.
# Answer: 0.65

Validation accuracy:
0.645
Rounded accuracy:
0.65


Q5. Feature selection

In [8]:
features = list(df_train.columns)
feature_scores = []

for feature in features:
    remaining_features = [f for f in features if f != feature]

    train_dicts = df_train[remaining_features].to_dict(orient="records")
    val_dicts = df_val[remaining_features].to_dict(orient="records")

    dv_without = DictVectorizer(sparse=False)
    X_train_without = dv_without.fit_transform(train_dicts)
    X_val_without = dv_without.transform(val_dicts)

    model_without = LogisticRegression(
        solver="liblinear", C=1.0, max_iter=1000, random_state=42
    )
    model_without.fit(X_train_without, y_train)
    y_pred = model_without.predict(X_val_without)
    score = accuracy_score(y_val, y_pred)

    # Compare accuracies before rounding.
    difference = original_accuracy - score
    feature_scores.append((feature, score, difference))

feature_results = pd.DataFrame(
    feature_scores, columns=["removed_feature", "accuracy", "difference"]
)
print("Accuracy without each feature:")
print(feature_results)

options = ["lead_source", "number_of_courses_viewed", "interaction_count"]
option_results = feature_results[feature_results["removed_feature"].isin(options)]
least_useful = option_results.loc[option_results["difference"].idxmin(), "removed_feature"]
print("Smallest difference among the options:")
print(least_useful)

# Differences: lead_source = 0.003, number_of_courses_viewed = 0.002,
# interaction_count = 0.044.
# Answer: number_of_courses_viewed

Accuracy without each feature:
            removed_feature  accuracy  difference
0               lead_source     0.642       0.003
1                  industry     0.645       0.000
2         employment_status     0.645       0.000
3                  location     0.645       0.000
4             annual_income     0.724      -0.079
5  number_of_courses_viewed     0.643       0.002
6         interaction_count     0.601       0.044
7                lead_score     0.644       0.001
Smallest difference among the options:
number_of_courses_viewed


Q6. Parameter tuning

In [9]:
# X_train and X_val still contain all features from Q4.
c_scores = []

print("C and validation accuracy:")
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    model_reg = LogisticRegression(
        solver="liblinear", C=C, max_iter=1000, random_state=42
    )
    model_reg.fit(X_train, y_train)
    y_pred = model_reg.predict(X_val)
    score = accuracy_score(y_val, y_pred)
    rounded_score = round(score, 3)
    c_scores.append((C, rounded_score))
    print(C, rounded_score)

# If rounded scores are equal, choose the smaller C.
best_c = min(c_scores, key=lambda result: (-result[1], result[0]))[0]
print("Best C:")
print(best_c)

# The best validation accuracy is 0.645 with C = 0.001.
# Answer: 0.001

C and validation accuracy:
1e-06 0.598
1e-05 0.598
0.0001 0.613
0.001 0.645
Best C:
0.001
